In [ ]:
!nvidia-smi

Mon Sep 21 19:39:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!gzip -dk /content/chembl.csv.gz
!ls -lh /content/chembl.csv
!wc -l /content/chembl.csv
!sha256sum /content/chembl.csv

-rw-r--r-- 1 root root 1.6G Sep 21 19:35 /content/chembl.csv
2750203 /content/chembl.csv
f81721223fb64e0432b89dfca87abbf5b9b6ab246700f1a365dc0667fdf5041e  /content/chembl.csv


In [ ]:
# ============================================================
# ChEMBL Heterogeneous Hardware Benchmark
# Google Colab / NVIDIA Tesla T4
# ============================================================

import time
import hashlib
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    LSTM,
    Dropout,
    Dense,
)
from tensorflow.keras.optimizers import Adam


# ============================================================
# Configuration
# ============================================================

DATA_PATH = "/content/chembl.csv"

RANDOM_SEED = 42
BENCHMARK_SIZE = 100_000
BATCH_SIZE = 128
WARMUP_EPOCHS = 1
TIMED_EPOCHS = 3

# Fixed sequence length for cross-hardware comparability
MAX_LENGTH = 404


# ============================================================
# Environment
# ============================================================

print("TensorFlow:", tf.__version__)
print("Physical devices:", tf.config.list_physical_devices())

tf.keras.utils.set_random_seed(RANDOM_SEED)


# ============================================================
# Read ChEMBL
# ============================================================

print("\nReading ChEMBL data...")

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    usecols=["Smiles", "AlogP"],
    low_memory=False,
)

df = df.dropna(subset=["Smiles", "AlogP"]).copy()

df["Smiles"] = df["Smiles"].astype(str)
df["AlogP"] = pd.to_numeric(df["AlogP"], errors="coerce")

df = df.dropna(subset=["AlogP"]).copy()

print("Clean observations:", len(df))


# ============================================================
# Deterministic benchmark sample
# ============================================================

df = df.sample(
    n=BENCHMARK_SIZE,
    random_state=RANDOM_SEED,
).reset_index(drop=True)

print("Benchmark observations:", len(df))


# ============================================================
# Character vocabulary
# ============================================================

smiles = df["Smiles"].tolist()

characters = sorted(set("".join(smiles)))

char_to_int = {
    char: index + 1
    for index, char in enumerate(characters)
}

VOCAB_SIZE = len(char_to_int) + 1

print("SMILES characters:", len(characters))
print("Vocabulary size including padding:", VOCAB_SIZE)
print("Maximum sequence length:", MAX_LENGTH)


# ============================================================
# Encode SMILES
# ============================================================

encoded_smiles = [
    [char_to_int[char] for char in smile]
    for smile in smiles
]

X = pad_sequences(
    encoded_smiles,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post",
)

y = df["AlogP"].to_numpy(dtype=np.float64)

print("Encoded X shape:", X.shape)
print("Target y shape:", y.shape)


# ============================================================
# Train/test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


# ============================================================
# Standardize target
# ============================================================

scaler = StandardScaler()

y_train_scaled = scaler.fit_transform(
    y_train.reshape(-1, 1)
).flatten()

y_test_scaled = scaler.transform(
    y_test.reshape(-1, 1)
).flatten()

print("Training target mean:", y_train_scaled.mean())
print("Training target std:", y_train_scaled.std())


# ============================================================
# Reproducibility fingerprints
# ============================================================

def array_hash(array):
    return hashlib.sha256(array.tobytes()).hexdigest()


print("\nReproducibility fingerprints:")
print("X_train:", array_hash(X_train))
print("X_test:", array_hash(X_test))
print("y_train:", array_hash(y_train))
print("y_test:", array_hash(y_test))


# ============================================================
# Model
# ============================================================

model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
    ),
    Bidirectional(
        LSTM(
            256,
            return_sequences=True,
        )
    ),
    Dropout(0.2),
    Bidirectional(
        LSTM(256)
    ),
    Dropout(0.2),
    Dense(1),
])

model.compile(
    optimizer=Adam(
        learning_rate=1e-4,
        clipnorm=1.0,
    ),
    loss="mse",
)

model.build(input_shape=(None, MAX_LENGTH))

model.summary()


# ============================================================
# Warm-up epoch
# ============================================================

print("\nWARM-UP EPOCH — not included in benchmark")

model.fit(
    X_train,
    y_train_scaled,
    batch_size=BATCH_SIZE,
    epochs=WARMUP_EPOCHS,
    validation_split=0.10,
    verbose=1,
)


# ============================================================
# Timed epochs
# ============================================================

training_samples = int(len(X_train) * 0.90)

epoch_times = []
throughputs = []

print("\nSTARTING TIMED BENCHMARK")

for epoch in range(TIMED_EPOCHS):

    start_time = time.perf_counter()

    history = model.fit(
        X_train,
        y_train_scaled,
        batch_size=BATCH_SIZE,
        epochs=1,
        validation_split=0.10,
        verbose=1,
    )

    elapsed = time.perf_counter() - start_time

    throughput = training_samples / elapsed

    epoch_times.append(elapsed)
    throughputs.append(throughput)

    print(
        f"\nTimed epoch {epoch + 1}: "
        f"{elapsed:.2f} s | "
        f"{throughput:.2f} samples/s | "
        f"loss={history.history['loss'][-1]:.6f} | "
        f"val_loss={history.history['val_loss'][-1]:.6f}"
    )


# ============================================================
# Final benchmark summary
# ============================================================

mean_time = np.mean(epoch_times)
median_time = np.median(epoch_times)

mean_throughput = np.mean(throughputs)
median_throughput = np.median(throughputs)

print("\n========================================")
print("BENCHMARK RESULTS")
print("========================================")

for i, (elapsed, throughput) in enumerate(
    zip(epoch_times, throughputs),
    start=1,
):
    print(
        f"Epoch {i}: "
        f"{elapsed:.2f} s | "
        f"{throughput:.2f} samples/s"
    )

print()
print(f"Mean time: {mean_time:.2f} s")
print(f"Median time: {median_time:.2f} s")
print(f"Mean throughput: {mean_throughput:.2f} samples/s")
print(f"Median throughput: {median_throughput:.2f} samples/s")
print("========================================")

TensorFlow: 2.20.0
Physical devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Reading ChEMBL data...
Clean observations: 2677485
Benchmark observations: 100000
SMILES characters: 46
Vocabulary size including padding: 47
Maximum sequence length: 404
Encoded X shape: (100000, 404)
Target y shape: (100000,)
X_train shape: (80000, 404)
X_test shape: (20000, 404)
Training target mean: -1.7923440509548528e-16
Training target std: 1.0

Reproducibility fingerprints:
X_train: 37aac87222e853bb5a7f6eca258043e966f15a2a9342a96ba7087688d85c0b2c
X_test: b76d5974c68d85adac7699f40f2fd593ed55a1409000be080e22917a8cf97617
y_train: 2a3e52c3227351d161f7a55b5d9ec27ffd03cec373d8a01cac5987598aa6dc69
y_test: 21301ff3c54b5ed186e40d6df58c4816e9943c9f82c80be6bd70a7390a762e23


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 404, 128)       │         6,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 404, 512)       │       788,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 404, 512)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 512)            │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,369,921 (9.04 MB)

 Trainable params: 2,369,921 (9.04 MB)

 Non-trainable params: 0 (0.00 B)


WARM-UP EPOCH — not included in benchmark
563/563 ━━━━━━━━━━━━━━━━━━━━ 50s 78ms/step - loss: 0.2635 - val_loss: 0.1823

STARTING TIMED BENCHMARK
563/563 ━━━━━━━━━━━━━━━━━━━━ 43s 76ms/step - loss: 0.1552 - val_loss: 0.1258

Timed epoch 1: 43.10 s | 1670.41 samples/s | loss=0.155160 | val_loss=0.125776
563/563 ━━━━━━━━━━━━━━━━━━━━ 43s 76ms/step - loss: 0.1243 - val_loss: 0.1328

Timed epoch 2: 43.03 s | 1673.40 samples/s | loss=0.124311 | val_loss=0.132781
563/563 ━━━━━━━━━━━━━━━━━━━━ 43s 76ms/step - loss: 0.1156 - val_loss: 0.1034

Timed epoch 3: 43.04 s | 1672.85 samples/s | loss=0.115627 | val_loss=0.103445

BENCHMARK RESULTS
Epoch 1: 43.10 s | 1670.41 samples/s
Epoch 2: 43.03 s | 1673.40 samples/s
Epoch 3: 43.04 s | 1672.85 samples/s

Mean time: 43.06 s
Median time: 43.04 s
Mean throughput: 1672.22 samples/s
Median throughput: 1672.85 samples/s
